In [1]:
from numpy.random import default_rng
import numpy as np
###################################

from pathlib import Path # para trabajar con rutas
import sys #información del sistema

ROOT = Path.cwd() # variable ROOT = current working directory
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent #reemplazamos el valor de ROOT por su padre hasta encontrar src

sys.path.insert(0, str(ROOT))

####################################

In [2]:
ROOT
ROOT.parent.parent.parent


WindowsPath('c:/Users/JHOSSEP/Documents')

In [3]:
# exportamos las funciones
from src.simuladorBJPC import BJPCPlan, simulate_bjpc_exp_suffstats
from src.priors import GammaPrior, BetaPrior, BetaGammaPrior
from src.posterior import BetaGammaPosterior
from src.loss import WarrantyTargets, AcceptanceWeights, g_exp_fail_before_L
# from loss import base_sampling_cost, 

In [4]:
############################### DATA BJPC #####################################

# empiezo con 7 unidades del producto 1 y 7 unidades del porducto 2, el experimento termina en la 6ta falla, 
# después de la primera falla retiro una unidad adicional 

plan = BJPCPlan(n=7, k=6, R=[1,0,0,0,0])
rng = default_rng(123)

data = simulate_bjpc_exp_suffstats(plan, lambda1=0.06, lambda2=0.03, rng=rng)
data

# hubo k1 fallas del producto 1, k2 fallas del producto 2. t_end es el tiempo en el que ocurrió la ultima falla

{'k1': 4, 'k2': 2, 'u': 73.51248704224477, 't_end': 30.2922991916515}

In [5]:
################################## PRIOR #####################################
gamma_prior = GammaPrior(a0=2.0, b0=1.5)
beta_prior  = BetaPrior(a1=2.0, a2=3.0)

bg_prior = BetaGammaPrior(gamma_prior=gamma_prior, beta_prior=beta_prior)

lambda1, lambda2 = bg_prior.sample(size=4)
print(lambda1, lambda2)

[1.01186887 0.27147963 0.04710456 0.32818171] [0.55840553 0.12385139 0.10509392 1.67080945]


In [6]:
############################### POSTERIOR #####################################
# usamos la fórmmula de la posterior actualizada ya conocida ya que es una conjugada y muestramos lamda1 y lambda2 a partir de la posterior
post = BetaGammaPosterior(
    a0=2.0, b0=1.0,
    a1=1.0, a2=1.0,
    k1=7, k2=3, u=120.5
)

# Muestrear lambda1, lambda2 
rng = default_rng(123)
lambda1_s, lambda2_s = post.sample(size=10000, rng=rng)

print(lambda1_s.mean(), lambda2_s.mean())

0.06591128395022802 0.03313578862001775


In [7]:
############################### LOSS #####################################
#Pero una acción no se toma por las tasas si no por el costo esperado de cada acción
# aceptamos o rechazamos el lote?



targets = WarrantyTargets(L1=100.0, L2=100.0)     # vida/garantía
weights = AcceptanceWeights(C1=200.0, C2=200.0)   # costo por falla temprana
# si el producto 1 falla antes de su vida objetivo, le asignas un costo 200.
# lo mismo para el producto 2. (C1 y C2)

#cálculo de g para cada posterior
g_s = g_exp_fail_before_L(
    lambda1=lambda1_s,
    lambda2=lambda2_s,
    targets=targets,
    weights=weights
)

phi = float(g_s.mean())
phi

# phi es la pérdida posterior esperada, o el costo posterior esperado asociado al desempeño del lote respecto a la garantía.

380.3686329019304

In [ ]:
############################### DECISION #####################################
"""
Aceptas si 𝜙 < 𝐶𝑟
Rechazas si 𝜙 ≥ 𝐶𝑟
"""
print

#Cr = 150.0
#accept = phi < Cr

#print("phi =", phi, "| decision =", "ACCEPT" if accept else "REJECT")


# Rechazo el lote, en esta oportunidad solo estamos evaluando 1 lote 

phi = 380.3686329019304 | decision = REJECT


In [9]:
############################### RIESGO DE BAYES #####################################
# “Si uso esta regla de decisión y este plan (n,k,R), ¿cuánto me cuesta EN PROMEDIO antes de ver datos?” 
# así elegimos el mejor plan (n,k,R) y la mejor regla de decisión (Cr) antes de ver datos, minimizando el riesgo de Bayes previo a la experimentación.